<a href="https://colab.research.google.com/github/gautampariniti00-hub/Email-Spam-Detection/blob/main/email_spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shantanudhakadd/email-spam-detection-dataset-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'email-spam-detection-dataset-classification' dataset.
Path to dataset files: /kaggle/input/email-spam-detection-dataset-classification


## Data Preprocessing

First, we'll load the dataset and perform initial cleaning and display its information.

In [ ]:
import pandas as pd
import os

# Construct the full path to the CSV file
csv_file_path = os.path.join(path, 'spam.csv')

# Load the dataset
df = pd.read_csv(csv_file_path, encoding='latin-1')

# Display the first 5 rows and information about the dataframe
display(df.head())
display(df.info())

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


None

Now that the dataset is loaded, we will clean the column names by dropping irrelevant columns and renaming the essential ones for clarity and ease of use.

In [ ]:
# Drop the 'Unnamed' columns
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

# Rename 'v1' to 'target' and 'v2' to 'text'
df.rename(columns={'v1': 'target', 'v2': 'text'}, inplace=True)

display(df.head())
display(df.info())

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


None

Now, let's check for duplicate rows and missing values. We will remove any duplicate entries and ensure there are no missing values that could affect our analysis.

In [ ]:
# Check for duplicates
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

# Drop duplicate rows
df.drop_duplicates(inplace=True)

# Check for missing values
missing_values = df.isnull().sum()
print("\nMissing values per column:")
print(missing_values)

display(df.info())

Number of duplicate rows: 403

Missing values per column:
target    0
text      0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  5169 non-null   object
 1   text    5169 non-null   object
dtypes: object(2)
memory usage: 121.1+ KB


None

Now that the columns are correctly cleaned and renamed, we will convert the 'target' column to numerical values. This step is essential for machine learning models that require numerical input.

Now, let's preprocess the text data. This involves lowercasing, tokenization, removing special characters and stop words, and stemming. We'll create a new column, `transformed_text`, for the processed messages.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

nltk.download('stopwords')

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def transform_text(text):
    text = text.lower() # 1. Lowercase
    text = nltk.word_tokenize(text) # 2. Tokenization

    y = []
    for i in text:
        if i.isalnum(): # 3. Removing special characters
            y.append(i)

    text = y[:]
    y.clear()
    for i in text:
        if i not in stop_words and i not in string.punctuation: # 4. Removing stop words and punctuation
            y.append(i)

    text = y[:]
    y.clear()
    for i in text:
        y.append(stemmer.stem(i)) # 5. Stemming

    return " ".join(y)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Convert 'target' column to numerical values
df['target'] = df['target'].map({'ham': 0, 'spam': 1})

display(df.head())
display(df['target'].value_counts())

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


,count
target,
0,4516
1,653


## Data Splitting

With the text transformed, we can now split the data into training and testing sets. This will allow us to train our model on one portion of the data and evaluate its performance on unseen data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


# df['target']=df['target'].map({'ham':0, 'spam': 1}) # This line was executed in a previous cell (639a424e) and causes NaNs if run again.

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

X = tfidf.fit_transform(df['text'])
y = df['target']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

print(f"y_test shape: {y_test.shape}")

print("\nDistribution of target in training set:")
print(y_train.value_counts(normalize=True))

print("\nDistribution of target in testing set:")
print(y_test.value_counts(normalize=True))

X_train shape: (4135, 5000)
X_test shape: (1034, 5000)
y_train shape: (4135,)
y_test shape: (1034,)

Distribution of target in training set:
target
0    0.873761
1    0.126239
Name: proportion, dtype: float64

Distribution of target in testing set:
target
0    0.873308
1    0.126692
Name: proportion, dtype: float64


## Linear Model Creation, Testing, and Evaluation

Now, we'll convert the text data into numerical features using TF-IDF, train a Logistic Regression model, make predictions, and evaluate its performance using accuracy, precision, and recall.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

# X_train and X_test are already TF-IDF transformed from previous steps

# Train a Logistic Regression model
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Accuracy: 0.9487
Precision: 0.9875
Recall: 0.6031


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer

# Re-initialize and re-fit TfidfVectorizer to ensure it's in a fitted state.
# This uses the same parameters (stop_words='english', max_features=5000)
# and fits on df['text'] as was done during the model training.
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf.fit(df['text'])

example_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim your prize!",
    "Hey, let's meet up for coffee tomorrow afternoon.",
    "URGENT! Your bank account has been compromised. Verify your details now.",
    "Reminder: Your appointment is scheduled for next Monday at 10 AM."
]

print("\n--- Testing with Example Messages ---")
for i, msg in enumerate(example_messages):
    # Preprocess the message
    transformed_msg = transform_text(msg)

    # Vectorize the message
    vectorized_msg = tfidf.transform([transformed_msg])

    # Make prediction
    prediction = model.predict(vectorized_msg)[0]

    # Get prediction probabilities
    proba = model.predict_proba(vectorized_msg)[0]

    print(f"\nMessage {i+1}: {msg}")
    if prediction == 1:
        print(f"Prediction: SPAM (Probability: {proba[1]:.4f})")
    else:
        print(f"Prediction: HAM (Probability: {proba[0]:.4f})")


--- Testing with Example Messages ---

Message 1: Congratulations! You've won a free iPhone. Click here to claim your prize!
Prediction: SPAM (Probability: 0.8333)

Message 2: Hey, let's meet up for coffee tomorrow afternoon.
Prediction: HAM (Probability: 0.9663)

Message 3: URGENT! Your bank account has been compromised. Verify your details now.
Prediction: HAM (Probability: 0.6650)

Message 4: Reminder: Your appointment is scheduled for next Monday at 10 AM.
Prediction: HAM (Probability: 0.9064)


## Interactive Prediction Interface

Let's create a simple user interface to input custom messages and see the model's prediction (spam or ham).

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def predict_spam_ham(text_input):
    # Preprocess the message
    transformed_msg = transform_text(text_input)

    # Vectorize the message
    vectorized_msg = tfidf.transform([transformed_msg])

    # Make prediction
    prediction = model.predict(vectorized_msg)[0]

    # Get prediction probabilities
    proba = model.predict_proba(vectorized_msg)[0]

    if prediction == 1:
        print(f"Prediction: SPAM (Probability: {proba[1]:.4f})")
    else:
        print(f"Prediction: HAM (Probability: {proba[0]:.4f})")

# Create a text input widget
text_widget = widgets.Textarea(
    value='',
    placeholder='Type your message here...',
    description='Message:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='100px')
)

# Create a button widget
predict_button = widgets.Button(
    description='Predict',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to predict if the message is spam or ham',
    icon='check'
)

# Create an output widget
output_widget = widgets.Output()

# Define a function to link button click to output
def on_button_click(b):
    with output_widget:
        output_widget.clear_output()
        predict_spam_ham(text_widget.value)

# Link the button's click event to the function
predict_button.on_click(on_button_click)

# Display the widgets
display(text_widget, predict_button, output_widget)

Textarea(value='', description='Message:', layout=Layout(height='100px', width='auto'), placeholder='Type your…

Button(description='Predict', icon='check', style=ButtonStyle(), tooltip='Click to predict if the message is s…

Output()